# 问题一：特征提取

# ex.ipynb

In [ ]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import skew, kurtosis
import os
import glob
from tqdm import tqdm

# =============================================================================
# 1. 特征提取核心函数 (已更新)
# =============================================================================
def calculate_all_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有需要的基础和高级特征。
    """
    # --- 基础时域特征 ---
    n = len(signal)
    mean_abs = np.mean(np.abs(signal))
    rms = np.sqrt(np.mean(signal**2))
    
    shape_factor = rms / mean_abs if mean_abs != 0 else 0
    impulse_factor = np.max(np.abs(signal)) / mean_abs if mean_abs != 0 else 0
    clearance_factor = np.max(np.abs(signal)) / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    crest_factor = np.max(np.abs(signal)) / rms if rms != 0 else 0
    margin_factor = clearance_factor
    
    all_features = {
        'Mean': np.mean(signal), 'Mean_abs': mean_abs, 'Var': np.var(signal),
        'Std': np.std(signal), 'Kurt': kurtosis(signal), 'Skew': skew(signal),
        'RMS': rms, 'Crest': crest_factor, 'Shape': shape_factor,
        'Impulse': impulse_factor, 'Margin': margin_factor, 'Clearance': clearance_factor,
        'P2P': np.max(signal) - np.min(signal)
    }

    # --- 频域特征 ---
    n_points = len(signal)
    fr = rpm / 60.0
    
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])
    
    # **【新增】: 计算您需要的4个基础频域特征**
    power_spectrum = amplitudes**2
    power_spectrum_normalized = power_spectrum / np.sum(power_spectrum) if np.sum(power_spectrum) != 0 else power_spectrum
    
    freq_mean = np.sum(freqs * power_spectrum_normalized)
    freq_std = np.sqrt(np.sum(((freqs - freq_mean)**2) * power_spectrum_normalized))
    epsilon = 1e-10
    freq_skew = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**3 * power_spectrum_normalized)
    freq_kurt = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**4 * power_spectrum_normalized) - 3

    all_features['FreqMean'] = freq_mean
    all_features['FreqSTD'] = freq_std
    all_features['FreqSkew'] = freq_skew
    all_features['FreqKurt'] = freq_kurt
    
    # --- 高级频域特征 ---
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # Part 1: 边带分析
    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        all_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        all_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        all_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        all_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # Part 2: 包络解调分析
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return all_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        all_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        all_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        all_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return all_features

# =============================================================================
# 2. 主执行函数 (逻辑保持不变)
# =============================================================================
def process_flat_directory(root_dir: str, rpm_value: float, sampling_rate: int):
    """
    遍历指定的单个文件夹，使用固定的RPM值，并提取所有特征。
    """
    if not os.path.isdir(root_dir):
        print(f"❌ 错误：提供的路径 '{root_dir}' 不是一个有效的文件夹。")
        return

    mat_files = sorted(glob.glob(os.path.join(root_dir, '*.mat')))
    if not mat_files:
        print(f"❌ 在文件夹 '{root_dir}' 中未找到任何 .mat 文件。")
        return

    print(f"✅ 在文件夹 '{root_dir}' 中找到 {len(mat_files)} 个 .mat 文件，开始处理...")
    
    all_results = []
    
    for file_path in tqdm(mat_files, desc="Processing .mat files"):
        try:
            base_name = os.path.basename(file_path)
            signal_key, _ = os.path.splitext(base_name)
            
            data = scipy.io.loadmat(file_path)
            
            if signal_key in data:
                signal = data[signal_key].flatten()
                
                features = calculate_all_features(signal, rpm_value, sampling_rate)
                
                features['SourceFile'] = base_name
                features['RPM'] = rpm_value
                
                all_results.append(features)
            else:
                print(f"\n警告: 在文件 '{base_name}' 中未找到预期的变量名 '{signal_key}'，已跳过。")

        except Exception as e:
            print(f"\n警告: 处理文件 {os.path.basename(file_path)} 时出错: {e}")

    if not all_results:
        print("\n扫描完成，但未能从任何文件中提取有效特征。")
        return

    final_df = pd.DataFrame(all_results)
    
    id_cols = ['SourceFile', 'RPM']
    feature_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + feature_cols]
    
    output_filename = '../data/features/target/final_features_complete.csv'
    final_df.to_csv(output_filename, index=False)
    
    print(f"\n\n🎉 --- 处理完成 --- 🎉")
    print(f"所有特征已汇总并保存到文件: '{output_filename}'")
    print("\n最终输出数据预览:")
    print(final_df.head().to_string())


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    TARGET_DIRECTORY = '../data/raw/target' 
    RPM_VALUE = 600
    SAMPLING_RATE = 32000
    
    # ======================= 配置结束 ===========================
    
    process_flat_directory(
        root_dir=TARGET_DIRECTORY,
        rpm_value=RPM_VALUE, 
        sampling_rate=SAMPLING_RATE
    )

# 2. 源域故障诊断

## 2.1 数据预处理

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTENC

def augment_with_smotenc(file_path: str, label_cols: list, categorical_feature_cols: list, min_samples: int):
    """
    使用SMOTENC对混合了数值和类别特征的数据集进行增强。
    此版本【不】进行独热编码。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 分离特征(X)和多列标签(y) ---
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中。请检查列表: {label_cols}")
        return None
        
    y = df[label_cols]
    X = df.drop(columns=label_cols)

    # --- 步骤 3: 创建临时的“组合标签” ---
    y_combined = y.astype(str).agg('_'.join, axis=1)
    
    print("\n--- 增强前各类别的样本数量 ---")
    class_counts_before = y_combined.value_counts()
    print(class_counts_before.to_string())

    # --- 步骤 4: 确定需要增强的类别和目标数量 ---
    sampling_strategy = {
        cls: min_samples for cls, count in class_counts_before.items() if count < min_samples
    }

    if not sampling_strategy:
        print("\n✅ 所有类别的样本数均已达到或超过目标数量，无需增强。")
        return df

    print(f"\n🔄 将对以下 {len(sampling_strategy)} 个类别的样本数量增强至 {min_samples}...")
    
    # --- 步骤 5: 应用SMOTENC ---
    try:
        # **核心步骤**: 找到类别型特征在X中的列索引位置
        actual_categorical_cols = [col for col in categorical_feature_cols if col in X.columns]
        categorical_features_indices = [X.columns.get_loc(col) for col in actual_categorical_cols]
        print(f"   - 已识别出以下 {len(categorical_features_indices)} 个类别型特征列用于SMOTENC处理:\n     {actual_categorical_cols}")

        min_class_count = class_counts_before.min()
        k_neighbors = min(min_class_count - 1, 5)
        
        if k_neighbors < 1:
            print("❌ SMOTENC执行失败: 数据集中存在只有一个样本的类别，无法进行操作。")
            return None

        smotenc = SMOTENC(categorical_features=categorical_features_indices, 
                          sampling_strategy=sampling_strategy, 
                          random_state=42, 
                          k_neighbors=k_neighbors)
                          
        X_resampled, y_combined_resampled = smotenc.fit_resample(X, y_combined)
        
    except Exception as e:
        print(f"\n❌ SMOTENC执行失败: {e}")
        return None

    # --- 步骤 6: 将增强后的“组合标签”拆分回原始的多列 ---
    y_resampled_split = y_combined_resampled.str.split('_', expand=True)
    y_resampled_split.columns = label_cols

    # --- 步骤 7: 合并增强后的特征和标签 ---
    # 将numpy数组X_resampled转回DataFrame，并保持原始列名
    final_df = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), y_resampled_split], axis=1)

    print("\n--- 增强后各类别的样本数量 ---")
    print(final_df[label_cols].astype(str).agg('_'.join, axis=1).value_counts().to_string())

    return final_df


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    # 1. 指定您的完整特征数据集文件名
    DATASET_FILE = '../data/features/dataset_complete.csv'
    
    # 2. 指定您希望组合起来进行均衡的【标签列】
    #    (根据您的代码截图，我更新为了3个标签)
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 3. **重要**: 在这里列出【所有】应该是类别型的【特征列】
    #    这些是除了标签之外的所有文本列。SMOTENC将对这些列进行特殊处理。
    CATEGORICAL_FEATURE_COLUMNS = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]

    # 4. 指定每个组合类别应有的【最少样本数】
    MIN_SAMPLES_PER_CLASS = 100

    # ======================= 配置结束 ===========================

    # 执行数据增强流程
    augmented_df = augment_with_smotenc(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_feature_cols=CATEGORICAL_FEATURE_COLUMNS,
        min_samples=MIN_SAMPLES_PER_CLASS
    )
    
    if augmented_df is not None:
        output_file = '../data/features/source/dataset_augmented.csv'
        augmented_df.to_csv(output_file, index=False)
        
        print(f"\n\n🎉 --- 数据增强完成 --- 🎉")
        print(f"✅ 增强后的数据集共有 {len(augmented_df)} 行。")
        print(f"✅ 列结构与输入文件完全一致。")
        print(f"✅ 已保存到新文件: '{output_file}'")
        print("\n--- 增强后数据集预览 ---")
        print(augmented_df.head().to_string())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

def final_preprocess_pipeline(file_path: str, label_cols: list, categorical_features: list, delete_cols: list = None):
    """
    一个完整且灵活的数据预处理流程，严格遵循用户定义。
    它将创建单列的“组合标签”以兼容后续的交叉验证。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 删除用户指定的列 ---
    if delete_cols:
        # 筛选出数据集中实际存在的、需要删除的列
        cols_to_drop_existing = [col for col in delete_cols if col in df.columns]
        df.drop(columns=cols_to_drop_existing, inplace=True)
        print(f"✅ 已删除指定列: {cols_to_drop_existing}。剩余 {df.shape[1]} 列。")

    # --- 步骤 3: 创建单列的“组合标签” ---
    # 这个步骤是为了解决后续交叉验证的报错问题
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中或已被删除。请检查列表: {label_cols}")
        return None
    
    # 将多个标签列合并成一个，用作后续的 y
    y = df[label_cols].astype(str).agg('_'.join, axis=1)
    # 从特征集中删除原始标签列
    X = df.drop(columns=label_cols)
    print(f"✅ 已成功将标签列 {label_cols} 合并为单列组合标签，用于分层抽样。")


    # --- 步骤 4: 数据类型处理和缺失值填充 ---
    print("\n🔄 正在处理特征集 X...")
    # 确定数值列（所有非明确指定的类别特征列）
    numeric_features = [col for col in X.columns if col not in categorical_features]
    for col in numeric_features:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        if X[col].isnull().any():
            X[col].fillna(X[col].median(), inplace=True)
            
    # 确定实际存在的类别特征列
    actual_categorical_features = [col for col in categorical_features if col in X.columns]
    for col in actual_categorical_features:
        if X[col].isnull().any():
            X[col].fillna('missing', inplace=True)

    # --- 步骤 5: 对类别型特征进行独热编码 ---
    print("\n🔄 正在对类别型特征进行独热编码...")
    X_encoded = pd.get_dummies(X, columns=actual_categorical_features, prefix=actual_categorical_features)
    print("✅ 特征集独热编码完成。")

    # --- 步骤 6: 数据集分割 ---
    print("\n🔄 正在划分数据集...")
    try:
        # 使用单列的组合标签 y 进行分层抽样
        X_train, X_test, y_train, y_test = train_test_split(
            X_encoded, y, 
            test_size=0.2, 
            random_state=42, 
            stratify=y
        )
        print("✅ 数据集已成功划分为训练集和测试集。")
    except Exception as e:
        print(f"❌ 数据集分割失败: {e}")
        return None

    # --- 步骤 7: 标准化数值型特征 ---
    print("\n🔄 正在标准化数值型特征...")
    scaler = StandardScaler()
    # 编码后，原始的数值列名仍然存在
    numeric_features_final = [col for col in numeric_features if col in X_train.columns]
    
    X_train[numeric_features_final] = scaler.fit_transform(X_train[numeric_features_final])
    X_test[numeric_features_final] = scaler.transform(X_test[numeric_features_final])
    print("✅ 标准化完成。")
    
    # --- 步骤 8: 使用pickle保存所有处理好的对象 ---
    processed_data_bundle = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'scaler': scaler
    }
    with open('../data/processed/processed_data.pkl - 1', 'wb') as f:
        pickle.dump(processed_data_bundle, f)

    print("\n\n🎉 --- 预处理流程全部完成 --- 🎉")
    print("✅ 所有处理好的数据和标准化模型已保存到 '../data/processed/processed_data.pkl - 1' 文件中。")
    print(f"   - 保存的 y_train 维度是: {y_train.shape}") # 确认y是单列
    
    return processed_data_bundle

# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    # 1. 指定您的完整数据集文件名
    DATASET_FILE = '../data/features/source/dataset_augmented.csv'

    
    # 2. 在这个列表中，输入您认为是【类别型】且需要独热编码的列名
    CATEGORICAL_COLUMNS_TO_ENCODE = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]
    
    # 3. 指定一个或多个列作为您的【标签】
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 4. 如果有需要删除的列，可以在这里指定
    DELETE_COLUMNS = ['SenLoc', 'HP', 'Freq', 'Len', 'OR_De_Type']
    # ======================= 配置结束 ===========================
    
    # **代码会自动处理配置中的逻辑冲突**
    # 例如，即使'PE'和'DeLoc'同时出现在LABEL_COLUMNS和CATEGORICAL_COLUMNS_TO_ENCODE中，
    # 代码也会优先将它们作为标签分离，而不会在特征集中对它们进行编码。
    # 同理，如果一个列同时出现在要编码和要删除的列表中，它会被优先删除。
    
    # 将用户定义的【标签列】从【类别特征】列表中排除，以确保逻辑清晰
    final_categorical_features = [
        col for col in CATEGORICAL_COLUMNS_TO_ENCODE if col not in LABEL_COLUMNS and col not in DELETE_COLUMNS
    ]

    final_data = final_preprocess_pipeline(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_features=final_categorical_features,
        delete_cols=DELETE_COLUMNS
    )
    
    if final_data:
        print("\n--- 预处理产物预览 ---")
        print(f"X_train 维度: {final_data['X_train'].shape}")
        print(f"y_train 维度: {final_data['y_train'].shape}")
        print("\nX_train 前5行预览:")
        print(final_data['X_train'].head().to_string())

## 2.2 训练

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 忽略一些可能出现的计算警告
warnings.filterwarnings('ignore')

# --- 绘图函数 ---

def plot_comparison_chart(results_df):
    """绘制模型比选的簇状条形图 (标题为英文)"""
    print("\n🔄 Generating model comparison chart...")
    
    train_acc = results_df['Train_Accuracy_Mean']
    validation_acc = results_df['Validation_Accuracy_Mean']
    model_names = results_df.index
    
    x = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, train_acc, width, label='Train Accuracy')
    rects2 = ax.bar(x + width/2, validation_acc, width, label='Validation Accuracy')

    ax.set_ylabel('Mean Accuracy')
    ax.set_title('Model Comparison on Train vs. Validation Sets')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names)
    ax.legend()
    ax.bar_label(rects1, padding=3, fmt='%.4f')
    ax.bar_label(rects2, padding=3, fmt='%.4f')
    ax.set_ylim(0, 1.1)

    fig.tight_layout()
    plt.savefig('../results/00_全流程/model_comparison_accuracy.png')
    print("✅ Comparison chart saved to 'model_comparison_accuracy.png'")
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    """计算并绘制标准化的混淆矩阵图 (采用手动标注数字)"""
    print("\n🔄 Generating confusion matrix chart...")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.5, str(cm[i, j]),
                    ha="center", va="center", color="black", size=10)

    ax.set_title(f'{model_name} Confusion Matrix on Test Set')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    
    plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_confusion_matrix.png')
    print(f"✅ Confusion matrix saved to '{model_name.replace(' ', '_').lower()}_confusion_matrix.png'")
    plt.show()

# --- 【新增】独立的特征重要性绘图函数 ---
def plot_feature_importance(model, feature_names, model_name):
    """
    为训练好的模型绘制特征重要性条形图。
    """
    print(f"\n🔄 Generating feature importance chart for {model_name}...")
    
    # 检查模型是否具有 feature_importances_ 属性
    if not hasattr(model, 'feature_importances_'):
        print(f"   - Model {model_name} does not have 'feature_importances_' attribute. Skipping plot.")
        return

    # 创建一个包含特征名和重要性得分的DataFrame
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    top_features = feature_importance_df

    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=top_features)
    
    plt.title(f'Top Feature Importances for {model_name}')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    
    plt.tight_layout()
    plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_feature_importance.png')
    print(f"✅ Feature importance chart saved to '{model_name.replace(' ', '_').lower()}_feature_importance.png'")
    plt.show()


# --- 主要实验函数 ---

def compare_models_via_cv(X_train, y_train, models, kfold):
    """通过交叉验证对多个模型进行性能比选"""
    print("\n--- Stage 1: Model Comparison via Cross-Validation ---")
    print("\n🔄 Starting cross-validation...")
    
    results = []
    for model_name, model in models.items():
        print(f"   - Evaluating: {model_name}...")
        cv_results = cross_validate(model, X_train, y_train, cv=kfold, 
                                    scoring=['accuracy'], n_jobs=-1,
                                    return_train_score=True)
        
        model_result = {'Model': model_name}
        model_result['Train_Accuracy_Mean'] = cv_results['train_accuracy'].mean()
        model_result['Validation_Accuracy_Mean'] = cv_results['test_accuracy'].mean()
        results.append(model_result)

    results_df = pd.DataFrame(results).set_index('Model')
    print("\n🎉 --- Cross-Validation Complete --- 🎉")
    print("Model Performance Comparison:")
    print(results_df.round(4).to_string())
    
    plot_comparison_chart(results_df)
    
    best_model_name = results_df['Validation_Accuracy_Mean'].idxmax()
    print(f"\n🏆 Based on mean validation accuracy, the best model is: **{best_model_name}**")
    return best_model_name

def train_and_evaluate_final_model(model, model_name: str, X_train, y_train, X_test, y_test, le):
    """在全部训练集上训练指定模型，在测试集上评估，并展示特征重要性"""
    print(f"\n--- Stage 2: Final Training and Evaluation of the Best Model ({model_name}) ---")
    
    print(f"\n🔄 Training the final {model_name} model on all training data...")
    model.fit(X_train, y_train)
    print("✅ Final model training complete.")

    print("\n🔄 Evaluating on the independent test set...")
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    print("\n--- Final Model Performance on Test Set ---")
    print(f"✅ Accuracy: {accuracy:.4f}")
    
    y_test_text = le.inverse_transform(y_test)
    y_pred_text = le.inverse_transform(y_pred)
    
    print("\n📋 Classification Report:")
    print(classification_report(y_test_text, y_pred_text))

    # 绘制混淆矩阵
    plot_confusion_matrix(y_test_text, y_pred_text, class_names=le.classes_, model_name=model_name)
    
    # **【新增】** 调用特征重要性绘图函数
    # 我们需要传递训练好的模型和特征的名字列表 (X_train.columns)
    plot_feature_importance(model, X_train.columns, model_name)
    
    # 保存模型
    model_filename = f'../models/00_全流程/{model_name.replace(" ", "_").lower()}_final_model.pkl'
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"\n✅ Trained model has been saved to: '{model_filename}'")


# --- 主执行函数 ---
if __name__ == "__main__":
    
    try:
        # **注意**：请确保这里的文件名正确
        with open('../data/processed/processed_data.pkl - 1', 'rb') as f:
            data_bundle = pickle.load(f)
        X_train = data_bundle['X_train']
        y_train_text = data_bundle['y_train']
        X_test = data_bundle['X_test']
        y_test_text = data_bundle['y_test']
        print(f"✅ Successfully loaded '../data/processed/processed_data.pkl - 1'.")
    except FileNotFoundError:
        print("❌ Error: Data file '../data/processed/processed_data.pkl - 1' not found.")
        exit()

    # --- 标签编码 ---
    le = LabelEncoder()
    le.fit(pd.concat([y_train_text, y_test_text]).unique())
    y_train_encoded = le.transform(y_train_text)
    y_test_encoded = le.transform(y_test_text)
    
    
    # --- 打印特征和标签列表 ---
    print("\n" + "="*50)
    print("--- Model Input Details ---")
    
    feature_list = X_train.columns.tolist()
    print(f"✅ The model will be trained on the following {len(feature_list)} features:")
    features_per_line = 5
    for i in range(0, len(feature_list), features_per_line):
        print("   - " + ", ".join(feature_list[i:i+features_per_line]))
        
    label_list = le.classes_.tolist()
    print(f"\n✅ The model will predict the following {len(label_list)} labels:")
    print(f"   - {label_list}")
    print("\nLabel to Integer Mapping:")
    print("   - " + str(dict(zip(le.classes_, le.transform(le.classes_)))))
    print("="*50)

    # --- 定义模型 ---
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    }

    # --- 执行实验 ---
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_model_name = compare_models_via_cv(X_train, y_train_encoded, models, kfold)
    train_and_evaluate_final_model(
        model=models[best_model_name],
        model_name=best_model_name,
        X_train=X_train,
        y_train=y_train_encoded,
        X_test=X_test,
        y_test=y_test_encoded,
        le=le
    )

# 3 迁移诊断

## 3.1 特征转换

In [ ]:
import pandas as pd
import numpy as np
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

def coral_align(Xs: np.ndarray, Xt: np.ndarray) -> np.ndarray:
    """
    对【已经标准化过】的Numpy数组执行CORAL算法。
    """
    print("\n🔄 Starting CORAL alignment on scaled data...")
    
    cov_s = np.cov(Xs, rowvar=False) + np.eye(Xs.shape[1]) * 1e-6
    cov_t = np.cov(Xt, rowvar=False) + np.eye(Xt.shape[1]) * 1e-6

    A_s = sqrtm(np.linalg.inv(cov_s))
    B_t = sqrtm(cov_t)
    
    transformation_matrix = np.dot(A_s, B_t)
    
    Xs_aligned_np = np.dot(Xs, transformation_matrix)
    Xs_aligned_np = np.real(Xs_aligned_np)
    
    print("✅ CORAL alignment complete.")
    return Xs_aligned_np

def plot_domain_distribution_tsne(Xs_scaled, Xt_scaled, Xs_aligned_scaled):
    """
    使用t-SNE降维，对比展示CORAL变换前后的数据分布。
    此版本会统一坐标轴范围。
    """
    print("\n🔄 Generating t-SNE distribution plots... (This may take a moment)")
    
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=300)
    
    # --- 变换前 ---
    combined_before = np.vstack((Xs_scaled, Xt_scaled))
    tsne_results_before = tsne.fit_transform(combined_before)
    Xs_tsne_before = tsne_results_before[:len(Xs_scaled)]
    Xt_tsne_before = tsne_results_before[len(Xs_scaled):]

    # --- 变换后 ---
    combined_after = np.vstack((Xs_aligned_scaled, Xt_scaled))
    tsne_results_after = tsne.fit_transform(combined_after)
    Xs_tsne_after = tsne_results_after[:len(Xs_aligned_scaled)]
    Xt_tsne_after = tsne_results_after[len(Xs_aligned_scaled):]

    # --- 绘图 ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    # 左图：变换前
    ax1.scatter(Xs_tsne_before[:, 0], Xs_tsne_before[:, 1], c='royalblue', label='Source Domain', alpha=0.7)
    ax1.scatter(Xt_tsne_before[:, 0], Xt_tsne_before[:, 1], c='darkorange', label='Target Domain', alpha=0.5)
    ax1.set_title('Before CORAL Alignment (t-SNE)')
    ax1.legend(); ax1.grid(True)

    # 右图：变换后
    ax2.scatter(Xs_tsne_after[:, 0], Xs_tsne_after[:, 1], c='royalblue', label='Source Domain', alpha=0.7)
    ax2.scatter(Xt_tsne_after[:, 0], Xt_tsne_after[:, 1], c='darkorange', label='Target Domain', alpha=0.5)
    ax2.set_title('After CORAL Alignment (t-SNE)')
    ax2.legend(); ax2.grid(True)

    # --- 【新增】统一坐标轴范围 ---
    # 1. 找到所有数据点x和y坐标的全局最小值和最大值
    all_x_coords = np.concatenate([tsne_results_before[:, 0], tsne_results_after[:, 0]])
    all_y_coords = np.concatenate([tsne_results_before[:, 1], tsne_results_after[:, 1]])
    
    x_min, x_max = all_x_coords.min(), all_x_coords.max()
    y_min, y_max = all_y_coords.min(), all_y_coords.max()
    
    # 2. 增加一点边距(padding)，让图像更好看
    x_padding = (x_max - x_min) * 0.05
    y_padding = (y_max - y_min) * 0.05
    
    # 3. 将统一的范围应用到两个子图上
    ax1.set_xlim(x_min - x_padding, x_max + x_padding)
    ax1.set_ylim(y_min - y_padding, y_max + y_padding)
    ax2.set_xlim(x_min - x_padding, x_max + x_padding)
    ax2.set_ylim(y_min - y_padding, y_max + y_padding)
    # --- 新增结束 ---

    fig.suptitle('Domain Distribution Comparison using t-SNE', fontsize=16)
    plt.savefig('../results/00_全流程/coral_tsne_comparison_unified_axes.png')
    print("✅ t-SNE chart with unified axes saved to '../results/00_全流程/coral_tsne_comparison_unified_axes.png'")
    plt.show()

# --- 主执行函数 ---
if __name__ == "__main__":
    
    SOURCE_DATA_FILE = '../data/features/dataset_complete.csv'
    TARGET_DATA_FILE = '../data/features/target/dataset_complete - 2.csv'
    FEATURE_COLUMNS = [
        'rpm', 'Mean', 'RMS', 'Var', 'Skew', 'Kurt', 'CF', 'MF', 'P2P', 
        'FreqMean', 'FreqSTD', 'FreqSkew', 'FreqKurt', 
        'IR_Sideband_Energy_DE', 'IR_Sideband_Ratio_DE', 'B_Sideband_Energy_DE', 'B_Sideband_Ratio_DE',
        'IR_Sideband_Energy_FE', 'IR_Sideband_Ratio_FE', 'B_Sideband_Energy_FE', 'B_Sideband_Ratio_FE',
        'Env_Peak_BPFO_DE', 'Env_Peak_BPFI_DE', 'Env_Peak_BSF_DE',
        'Env_Peak_BPFO_FE', 'Env_Peak_BPFI_FE', 'Env_Peak_BSF_FE'
    ]
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    try:
        source_df = pd.read_csv(SOURCE_DATA_FILE)
        target_df = pd.read_csv(TARGET_DATA_FILE)
    except FileNotFoundError as e:
        print(f"❌ 错误：文件未找到。请检查文件名: {e}"); exit()

    X_source = source_df[FEATURE_COLUMNS]
    y_source = source_df[LABEL_COLUMNS]
    X_target = target_df[FEATURE_COLUMNS]
    print(f"✅ 数据加载和筛选完成。")

    print("\n🔄 Applying consistent standardization...")
    scaler = StandardScaler()
    X_source_scaled = scaler.fit_transform(X_source)
    X_target_scaled = scaler.transform(X_target)
    
    X_source_aligned_scaled = coral_align(X_source_scaled, X_target_scaled)

    plot_domain_distribution_tsne(
        Xs_scaled=X_source_scaled, 
        Xt_scaled=X_target_scaled, 
        Xs_aligned_scaled=X_source_aligned_scaled
    )

    # aligned_source_dataset = pd.concat([y_source, pd.DataFrame(X_source_aligned_scaled, columns=FEATURE_COLUMNS)], axis=1)
    # aligned_source_dataset.to_csv('../data/features/target/source_data_aligned_scaled.csv', index=False)

## 3.2 增强

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE # <--- 关键修正：导入标准SMOTE

def augment_data(file_path: str, label_cols: list, min_samples: int):
    """
    使用标准SMOTE对纯数值特征的数据集进行增强。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 分离特征(X)和多列标签(y) ---
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中。请检查列表: {label_cols}")
        return None
        
    y = df[label_cols]
    X = df.drop(columns=label_cols)

    # --- 步骤 3: 创建临时的“组合标签” ---
    y_combined = y.astype(str).agg('_'.join, axis=1)
    
    print("\n--- 增强前各类别的样本数量 ---")
    class_counts_before = y_combined.value_counts()
    print(class_counts_before.to_string())

    # --- 步骤 4: 确定需要增强的类别和目标数量 ---
    sampling_strategy = {
        cls: min_samples for cls, count in class_counts_before.items() if count < min_samples
    }

    if not sampling_strategy:
        print("\n✅ 所有类别的样本数均已达到或超过目标数量，无需增强。")
        return df

    print(f"\n🔄 将对以下 {len(sampling_strategy)} 个类别的样本数量增强至 {min_samples}...")
    
    # --- 步骤 5: 应用标准SMOTE ---
    try:
        min_class_count = class_counts_before.min()
        k_neighbors = min(min_class_count - 1, 5)
        
        if k_neighbors < 1:
            print("❌ SMOTE执行失败: 数据集中存在只有一个样本的类别，无法进行操作。")
            return None

        # **关键修正**: 使用标准SMOTE，它不需要categorical_features参数
        smote = SMOTE(sampling_strategy=sampling_strategy, 
                      random_state=42, 
                      k_neighbors=k_neighbors)
                          
        X_resampled, y_combined_resampled = smote.fit_resample(X, y_combined)
        
    except Exception as e:
        print(f"\n❌ SMOTE执行失败: {e}")
        return None

    # --- 步骤 6: 将增强后的“组合标签”拆分回原始的多列 ---
    y_resampled_split = y_combined_resampled.str.split('_', expand=True)
    y_resampled_split.columns = label_cols

    # --- 步骤 7: 合并增强后的特征和标签 ---
    final_df = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), y_resampled_split], axis=1)

    print("\n--- 增强后各类别的样本数量 ---")
    print(final_df[label_cols].astype(str).agg('_'.join, axis=1).value_counts().to_string())

    return final_df


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    # 1. 指定经过CORAL处理后的、纯数值特征的数据集文件名
    DATASET_FILE = '../data/features/target/source_data_aligned.csv'
    
    # 2. 指定您希望组合起来进行均衡的【标签列】
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']
    
    # 3. 指定每个组合类别应有的【最少样本数】
    MIN_SAMPLES_PER_CLASS = 100
    
    # 您不再需要指定类别型特征列
    
    # ======================= 配置结束 ===========================

    augmented_df = augment_data(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        min_samples=MIN_SAMPLES_PER_CLASS
    )
    
    if augmented_df is not None:
        output_file = '../data/features/target/dataset_augmented - 2.csv'
        augmented_df.to_csv(output_file, index=False)
        
        print(f"\n\n🎉 --- 数据增强完成 --- 🎉")
        print(f"✅ 增强后的数据集共有 {len(augmented_df)} 行。")
        print(f"✅ 已保存到新文件: '{output_file}'")
        print("\n--- 增强后数据集预览 ---")
        print(augmented_df.head().to_string())

## 3.2 预处理

### 3.2.1 源域

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

def final_preprocess_pipeline(file_path: str, label_cols: list, categorical_features: list, delete_cols: list = None):
    """
    一个完整且灵活的数据预处理流程，严格遵循用户定义。
    它将创建单列的“组合标签”以兼容后续的交叉验证。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 删除用户指定的列 ---
    if delete_cols:
        # 筛选出数据集中实际存在的、需要删除的列
        cols_to_drop_existing = [col for col in delete_cols if col in df.columns]
        df.drop(columns=cols_to_drop_existing, inplace=True)
        print(f"✅ 已删除指定列: {cols_to_drop_existing}。剩余 {df.shape[1]} 列。")

    # --- 步骤 3: 创建单列的“组合标签” ---
    # 这个步骤是为了解决后续交叉验证的报错问题
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中或已被删除。请检查列表: {label_cols}")
        return None
    
    # 将多个标签列合并成一个，用作后续的 y
    y = df[label_cols].astype(str).agg('_'.join, axis=1)
    # 从特征集中删除原始标签列
    X = df.drop(columns=label_cols)
    print(f"✅ 已成功将标签列 {label_cols} 合并为单列组合标签，用于分层抽样。")


    # --- 步骤 4: 数据类型处理和缺失值填充 ---
    print("\n🔄 正在处理特征集 X...")
    # 确定数值列（所有非明确指定的类别特征列）
    numeric_features = [col for col in X.columns if col not in categorical_features]
    for col in numeric_features:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        if X[col].isnull().any():
            X[col].fillna(X[col].median(), inplace=True)
            
    # 确定实际存在的类别特征列
    actual_categorical_features = [col for col in categorical_features if col in X.columns]
    for col in actual_categorical_features:
        if X[col].isnull().any():
            X[col].fillna('missing', inplace=True)

    # --- 步骤 5: 对类别型特征进行独热编码 ---
    print("\n🔄 正在对类别型特征进行独热编码...")
    X_encoded = pd.get_dummies(X, columns=actual_categorical_features, prefix=actual_categorical_features)
    print("✅ 特征集独热编码完成。")

    # --- 步骤 6: 数据集分割 ---
    print("\n🔄 正在划分数据集...")
    try:
        # 使用单列的组合标签 y 进行分层抽样
        X_train, X_test, y_train, y_test = train_test_split(
            X_encoded, y, 
            test_size=0.2, 
            random_state=42, 
            stratify=y
        )
        print("✅ 数据集已成功划分为训练集和测试集。")
    except Exception as e:
        print(f"❌ 数据集分割失败: {e}")
        return None

    # --- 步骤 7: 标准化数值型特征 ---
    print("\n🔄 正在标准化数值型特征...")
    scaler = StandardScaler()
    # 编码后，原始的数值列名仍然存在
    numeric_features_final = [col for col in numeric_features if col in X_train.columns]
    
    X_train[numeric_features_final] = scaler.fit_transform(X_train[numeric_features_final])
    X_test[numeric_features_final] = scaler.transform(X_test[numeric_features_final])
    print("✅ 标准化完成。")
    
    # --- 步骤 8: 使用pickle保存所有处理好的对象 ---
    processed_data_bundle = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'scaler': scaler
    }
    with open('../data/processed/processed_data - 2.pkl', 'wb') as f:
        pickle.dump(processed_data_bundle, f)

    print("\n\n🎉 --- 预处理流程全部完成 --- 🎉")
    print("✅ 所有处理好的数据和标准化模型已保存到 '../data/processed/processed_data - 2.pkl' 文件中。")
    print(f"   - 保存的 y_train 维度是: {y_train.shape}") # 确认y是单列
    
    return processed_data_bundle

# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    # 1. 指定您的完整数据集文件名
    DATASET_FILE = '../data/features/target/dataset_augmented - 2.csv'

    
    # 2. 在这个列表中，输入您认为是【类别型】且需要独热编码的列名
    CATEGORICAL_COLUMNS_TO_ENCODE = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]
    
    # 3. 指定一个或多个列作为您的【标签】
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 4. 如果有需要删除的列，可以在这里指定
    DELETE_COLUMNS = ['SenLoc', 'HP', 'Freq', 'Len', 'OR_De_Type']
    # ======================= 配置结束 ===========================
    
    # **代码会自动处理配置中的逻辑冲突**
    # 例如，即使'PE'和'DeLoc'同时出现在LABEL_COLUMNS和CATEGORICAL_COLUMNS_TO_ENCODE中，
    # 代码也会优先将它们作为标签分离，而不会在特征集中对它们进行编码。
    # 同理，如果一个列同时出现在要编码和要删除的列表中，它会被优先删除。
    
    # 将用户定义的【标签列】从【类别特征】列表中排除，以确保逻辑清晰
    final_categorical_features = [
        col for col in CATEGORICAL_COLUMNS_TO_ENCODE if col not in LABEL_COLUMNS and col not in DELETE_COLUMNS
    ]

    final_data = final_preprocess_pipeline(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_features=final_categorical_features,
        delete_cols=DELETE_COLUMNS
    )
    
    if final_data:
        print("\n--- 预处理产物预览 ---")
        print(f"X_train 维度: {final_data['X_train'].shape}")
        print(f"y_train 维度: {final_data['y_train'].shape}")
        print("\nX_train 前5行预览:")
        print(final_data['X_train'].head().to_string())

### 3.2.2 目标域

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

def preprocess_target_domain(target_file_path: str, source_preprocessor_path: str, 
                             label_cols: list, categorical_features: list, delete_cols: list):
    """
    对目标域数据进行预处理，严格复用源域学到的规则和用户配置。
    """
    # --- 步骤 1: 加载源域的处理工具和元数据 ---
    try:
        with open(source_preprocessor_path, 'rb') as f:
            source_bundle = pickle.load(f)
        
        scaler = source_bundle['scaler']
        source_train_columns = source_bundle['X_train'].columns
        
        # 从已保存的X_train中推断出数值列的准确列表
        source_numeric_cols = [
            col for col in source_bundle['X_train'].columns 
            if source_bundle['X_train'][col].dtype != 'uint8'
        ]
        print(f"✅ 成功加载源域预处理工具 '{source_preprocessor_path}'。")

    except FileNotFoundError:
        print(f"❌ 错误：找不到源域预处理文件 '{source_preprocessor_path}'。请先运行源域的预处理脚本。")
        return None

    # --- 步骤 2: 加载目标域数据集 ---
    try:
        df_target = pd.read_csv(target_file_path)
        print(f"✅ 成功加载目标域数据 '{target_file_path}'。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{target_file_path}' 未找到。")
        return None

    # --- 步骤 3: 应用与源域完全相同的预处理步骤 ---
    
    # 3.1 删除用户指定的列
    # errors='ignore'确保即使列不存在也不会报错
    df_target.drop(columns=delete_cols, inplace=True, errors='ignore')
    
    # 3.2 从目标域数据中分离出特征（它没有真实标签）
    # 我们需要删除在源域中被当作标签的列，以保持特征集一致
    target_feature_cols = [col for col in df_target.columns if col not in label_cols]
    X_target = df_target[target_feature_cols]

    # 3.3 数据类型和缺失值处理
    print("\n🔄 正在处理数据类型和缺失值...")
    # 筛选出实际存在的类别和数值特征列
    actual_categorical_features = [col for col in categorical_features if col in X_target.columns]
    actual_numeric_features = [col for col in X_target.columns if col not in actual_categorical_features]

    for col in actual_numeric_features:
        X_target[col] = pd.to_numeric(X_target[col], errors='coerce')
        X_target[col].fillna(X_target[col].median(), inplace=True)
            
    for col in actual_categorical_features:
        X_target[col].fillna('missing', inplace=True)

    # 3.4 独热编码
    print("🔄 正在进行独热编码...")
    X_target_encoded = pd.get_dummies(X_target, columns=actual_categorical_features, prefix=actual_categorical_features)
    
    # 3.5 **关键步骤**: 列对齐
    # 使用源域训练集的列作为模板，确保目标域的列完全一致
    # fill_value=0 确保了如果目标域缺少某个类别，对应的列会被创建并填充为0
    X_target_aligned = X_target_encoded.reindex(columns=source_train_columns, fill_value=0)
    print("✅ 特征列已与源域训练集对齐。")
    
    # 3.6 **最关键步骤**: 标准化
    print("🔄 正在使用【源域的】scaler进行标准化...")
    # 只对数值列进行标准化
    # 确保只对目标域中实际存在的数值列进行操作
    numeric_cols_to_scale = [col for col in source_numeric_cols if col in X_target_aligned.columns]
    X_target_aligned[numeric_cols_to_scale] = scaler.transform(X_target_aligned[numeric_cols_to_scale])
    print("✅ 标准化完成。")
    
    # --- 步骤 4: 保存处理好的目标域特征 ---
    output_file = '../data/processed/processed_data - target.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump({'X_target': X_target_aligned}, f)
        
    print(f"\n\n🎉 --- 目标域预处理完成 --- 🎉")
    print(f"✅ 处理好的目标域特征已保存到 '{output_file}'。")
    
    return X_target_aligned

# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 (您的定义) =========================
    # 1. 指定您的目标域数据集文件名
    TARGET_DATASET_FILE = '../data/features/target/dataset_complete - 2.csv'
    
    # 2. 指定之前处理源域数据时生成的.pkl文件名
    SOURCE_PREPROCESSOR_FILE = '../data/processed/processed_data - 2.pkl'

    # 3. (从源域脚本复制) 您定义的【模型预测目标 (y)】列名
    #    这些列将从目标域特征集中被移除
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 4. (从源域脚本复制) 您定义的需要进行独热编码的【类别型特征 (X)】
    CATEGORICAL_FEATURES = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]
    
    # 5. (从源域脚本复制) 您定义的希望【彻底删除】的列名
    DELETE_COLUMNS = ['SenLoc', 'HP', 'Freq', 'Len', 'OR_De_Type']
    # ======================= 配置结束 ===========================

    # 代码会自动处理配置中的逻辑冲突
    # 例如，'PE'和'DeLoc'在CATEGORICAL_FEATURES中，但因为它们也是LABEL_COLUMNS，
    # 它们会先从特征集中被移除，因此不会参与后续的独热编码
    final_categorical_features_for_X = [
        col for col in CATEGORICAL_FEATURES if col not in LABEL_COLUMNS and col not in DELETE_COLUMNS
    ]

    processed_target_data = preprocess_target_domain(
        target_file_path=TARGET_DATASET_FILE,
        source_preprocessor_path=SOURCE_PREPROCESSOR_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_features=final_categorical_features_for_X,
        delete_cols=DELETE_COLUMNS
    )
    
    if processed_target_data is not None:
        print("\n--- 预处理产物预览 ---")
        print(f"X_target 维度: {processed_target_data.shape}")
        print("\nX_target (处理后) 前5行预览:")
        print(processed_target_data.head().to_string())

## 3.3 模型训练、比选、验证、诊断

### 仅有源域数据分布发生变化

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 忽略一些可能出现的计算警告
warnings.filterwarnings('ignore')

# --- 绘图函数 ---

def plot_comparison_chart(results_df):
    """绘制模型比选的簇状条形图 (标题为英文)"""
    print("\n🔄 Generating model comparison chart...")
    
    train_acc = results_df['Train_Accuracy_Mean']
    validation_acc = results_df['Validation_Accuracy_Mean']
    model_names = results_df.index
    
    x = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, train_acc, width, label='Train Accuracy')
    rects2 = ax.bar(x + width/2, validation_acc, width, label='Validation Accuracy')

    ax.set_ylabel('Mean Accuracy')
    ax.set_title('Model Comparison on Train vs. Validation Sets')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names)
    ax.legend()
    ax.bar_label(rects1, padding=3, fmt='%.4f')
    ax.bar_label(rects2, padding=3, fmt='%.4f')
    ax.set_ylim(0, 1.1)

    fig.tight_layout()
    plt.savefig('../results/00_全流程/model_comparison_accuracy.png')
    print("✅ Comparison chart saved to 'model_comparison_accuracy.png'")
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    """计算并绘制标准化的混淆矩阵图 (采用手动标注数字)"""
    print("\n🔄 Generating confusion matrix chart...")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.5, str(cm[i, j]),
                    ha="center", va="center", color="black", size=10)

    ax.set_title(f'{model_name} Confusion Matrix on Test Set')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    
    plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_confusion_matrix.png')
    print(f"✅ Confusion matrix saved to '{model_name.replace(' ', '_').lower()}_confusion_matrix.png'")
    plt.show()

# --- 【新增】独立的特征重要性绘图函数 ---
def plot_feature_importance(model, feature_names, model_name):
    """
    为训练好的模型绘制特征重要性条形图。
    """
    print(f"\n🔄 Generating feature importance chart for {model_name}...")
    
    # 检查模型是否具有 feature_importances_ 属性
    if not hasattr(model, 'feature_importances_'):
        print(f"   - Model {model_name} does not have 'feature_importances_' attribute. Skipping plot.")
        return

    # 创建一个包含特征名和重要性得分的DataFrame
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    # 只展示最重要的前20个特征
    top_features = feature_importance_df.head(27)

    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=top_features)
    
    plt.title(f'Top 20 Feature Importances for {model_name}')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    
    plt.tight_layout()
    plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_feature_importance.png')
    print(f"✅ Feature importance chart saved to '{model_name.replace(' ', '_').lower()}_feature_importance.png'")
    plt.show()


# --- 主要实验函数 ---

def compare_models_via_cv(X_train, y_train, models, kfold):
    """通过交叉验证对多个模型进行性能比选"""
    print("\n--- Stage 1: Model Comparison via Cross-Validation ---")
    print("\n🔄 Starting cross-validation...")
    
    results = []
    for model_name, model in models.items():
        print(f"   - Evaluating: {model_name}...")
        cv_results = cross_validate(model, X_train, y_train, cv=kfold, 
                                    scoring=['accuracy'], n_jobs=-1,
                                    return_train_score=True)
        
        model_result = {'Model': model_name}
        model_result['Train_Accuracy_Mean'] = cv_results['train_accuracy'].mean()
        model_result['Validation_Accuracy_Mean'] = cv_results['test_accuracy'].mean()
        results.append(model_result)

    results_df = pd.DataFrame(results).set_index('Model')
    print("\n🎉 --- Cross-Validation Complete --- 🎉")
    print("Model Performance Comparison:")
    print(results_df.round(4).to_string())
    
    plot_comparison_chart(results_df)
    
    best_model_name = results_df['Validation_Accuracy_Mean'].idxmax()
    print(f"\n🏆 Based on mean validation accuracy, the best model is: **{best_model_name}**")
    return best_model_name

def train_and_evaluate_final_model(model, model_name: str, X_train, y_train, X_test, y_test, le):
    """在全部训练集上训练指定模型，在测试集上评估，并展示特征重要性"""
    print(f"\n--- Stage 2: Final Training and Evaluation of the Best Model ({model_name}) ---")
    
    print(f"\n🔄 Training the final {model_name} model on all training data...")
    model.fit(X_train, y_train)
    print("✅ Final model training complete.")

    print("\n🔄 Evaluating on the independent test set...")
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    print("\n--- Final Model Performance on Test Set ---")
    print(f"✅ Accuracy: {accuracy:.4f}")
    
    y_test_text = le.inverse_transform(y_test)
    y_pred_text = le.inverse_transform(y_pred)
    
    print("\n📋 Classification Report:")
    print(classification_report(y_test_text, y_pred_text))

    # 绘制混淆矩阵
    plot_confusion_matrix(y_test_text, y_pred_text, class_names=le.classes_, model_name=model_name)
    
    # **【新增】** 调用特征重要性绘图函数
    # 我们需要传递训练好的模型和特征的名字列表 (X_train.columns)
    plot_feature_importance(model, X_train.columns, model_name)
    
    # 保存模型
    model_filename = f'../models/00_全流程/{model_name.replace(" ", "_").lower()}_final_model.pkl'
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"\n✅ Trained model has been saved to: '{model_filename}'")


# --- 主执行函数 ---
if __name__ == "__main__":
    
    try:
        # **注意**：请确保这里的文件名正确
        with open('../data/processed/processed_data - 2.pkl', 'rb') as f:
            data_bundle = pickle.load(f)
        X_train = data_bundle['X_train']
        y_train_text = data_bundle['y_train']
        X_test = data_bundle['X_test']
        y_test_text = data_bundle['y_test']
        print(f"✅ Successfully loaded '../data/processed/processed_data.pkl - 1'.")
    except FileNotFoundError:
        print("❌ Error: Data file '../data/processed/processed_data.pkl - 1' not found.")
        exit()

    # --- 标签编码 ---
    le = LabelEncoder()
    le.fit(pd.concat([y_train_text, y_test_text]).unique())
    y_train_encoded = le.transform(y_train_text)
    y_test_encoded = le.transform(y_test_text)
    
    
    # --- 打印特征和标签列表 ---
    print("\n" + "="*50)
    print("--- Model Input Details ---")
    
    feature_list = X_train.columns.tolist()
    print(f"✅ The model will be trained on the following {len(feature_list)} features:")
    features_per_line = 5
    for i in range(0, len(feature_list), features_per_line):
        print("   - " + ", ".join(feature_list[i:i+features_per_line]))
        
    label_list = le.classes_.tolist()
    print(f"\n✅ The model will predict the following {len(label_list)} labels:")
    print(f"   - {label_list}")
    print("\nLabel to Integer Mapping:")
    print("   - " + str(dict(zip(le.classes_, le.transform(le.classes_)))))
    print("="*50)

    # --- 定义模型 ---
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    }

    # --- 执行实验 ---
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_model_name = compare_models_via_cv(X_train, y_train_encoded, models, kfold)
    train_and_evaluate_final_model(
        model=models[best_model_name],
        model_name=best_model_name,
        X_train=X_train,
        y_train=y_train_encoded,
        X_test=X_test,
        y_test=y_test_encoded,
        le=le
    )

### RFECV

#### 五彩斑斓的图

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_selection import RFECV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

# 忽略一些可能出现的计算警告
warnings.filterwarnings('ignore')

# --- 绘图函数 ---
def plot_comparison_chart(results_df, title_suffix=""):
    print(f"\n🔄 Generating model comparison chart{title_suffix}...")
    train_acc = results_df['Train_Accuracy_Mean']
    validation_acc = results_df['Validation_Accuracy_Mean']
    model_names = results_df.index
    x = np.arange(len(model_names)); width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, train_acc, width, label='Train Accuracy')
    rects2 = ax.bar(x + width/2, validation_acc, width, label='Validation Accuracy')
    ax.set_ylabel('Mean Accuracy'); ax.set_title(f'Model Comparison on Train vs. Validation Sets{title_suffix}')
    ax.set_xticks(x); ax.set_xticklabels(model_names); ax.legend()
    ax.bar_label(rects1, padding=3, fmt='%.4f'); ax.bar_label(rects2, padding=3, fmt='%.4f')
    ax.set_ylim(0, 1.1); fig.tight_layout()
    plt.savefig(f'../results/00_全流程/model_comparison_accuracy{title_suffix.replace(" ", "_")}.png')
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    print("\n🔄 Generating confusion matrix chart...")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.5, str(cm[i, j]), ha="center", va="center", color="black", size=10)
    ax.set_title(f'{model_name} Confusion Matrix on Test Set'); ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_confusion_matrix.png')
    plt.show()

def plot_feature_importance(model, feature_names, model_name):
    print(f"\n🔄 Generating feature importance chart for {model_name}...")
    if not hasattr(model, 'feature_importances_'): return
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)
    top_features = feature_importance_df.head(20)
    plt.figure(figsize=(12, 8)); sns.barplot(x='Importance', y='Feature', data=top_features)
    plt.title(f'Top {len(top_features)} Feature Importances for {model_name}'); plt.xlabel('Importance Score'); plt.ylabel('Features')
    plt.tight_layout(); plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_feature_importance.png')
    plt.show()

# --- 主要实验函数 ---
def select_features_with_rfecv(X_train, y_train, model_for_selection, kfold):
    print("\n--- Stage 1: Wrapper-based Feature Selection ---")
    print(f"\n🔄 Starting RFECV with {type(model_for_selection).__name__}...")
    rfecv = RFECV(estimator=model_for_selection, step=1, cv=kfold, scoring="f1_weighted", min_features_to_select=5, n_jobs=-1)
    rfecv.fit(X_train, y_train)
    print(f"✅ RFECV complete. Optimal number of features found: {rfecv.n_features_}")
    selected_features = X_train.columns[rfecv.support_]
    print(f"   - Selected features are: \n{selected_features.tolist()}")
    X_train_selected = rfecv.transform(X_train)
    return pd.DataFrame(X_train_selected, index=X_train.index, columns=selected_features), selected_features

def compare_models_via_cv(X_train, y_train, models, kfold, title_suffix=""):
    print(f"\n--- Stage 2: Model Comparison{title_suffix} ---")
    results = []
    for model_name, model in models.items():
        cv_results = cross_validate(model, X_train, y_train, cv=kfold, scoring=['accuracy'], n_jobs=-1, return_train_score=True)
        model_result = {'Model': model_name, 'Train_Accuracy_Mean': cv_results['train_accuracy'].mean(), 'Validation_Accuracy_Mean': cv_results['test_accuracy'].mean()}
        results.append(model_result)
    results_df = pd.DataFrame(results).set_index('Model')
    print(f"\nModel Performance Comparison{title_suffix}:\n{results_df.round(4).to_string()}")
    plot_comparison_chart(results_df, title_suffix)
    best_model_name = results_df['Validation_Accuracy_Mean'].idxmax()
    print(f"\n🏆 Based on mean validation accuracy, the best model is: **{best_model_name}**")
    return best_model_name

def train_and_evaluate_final_model(model, model_name, X_train, y_train, X_test, y_test, le):
    print(f"\n--- Stage 3: Final Training and Evaluation ({model_name}) ---")
    model.fit(X_train, y_train)
    print("✅ Final model training complete.")
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n--- Final Model Performance on Test Set ---\n✅ Accuracy: {accuracy:.4f}")
    y_test_text, y_pred_text = le.inverse_transform(y_test), le.inverse_transform(y_pred)
    print("\n📋 Classification Report:"); print(classification_report(y_test_text, y_pred_text))
    plot_confusion_matrix(y_test_text, y_pred_text, class_names=le.classes_, model_name=model_name)
    plot_feature_importance(model, X_train.columns, model_name)
    model_filename = f'../models/00_全流程/{model_name.replace(" ", "_").lower()}_final_model.pkl'
    with open(model_filename, 'wb') as f: pickle.dump(model, f)
    print(f"\n✅ Trained model has been saved to: '{model_filename}'")
    return model

def predict_target_domain(model, model_name, target_data_path, selected_features, le):
    print("\n" + "="*50 + f"\n--- Stage 4: Predicting on Target Domain Data ---")
    try:
        with open(target_data_path, 'rb') as f:
            target_data_bundle = pickle.load(f)
        X_target = target_data_bundle['X_target']
        print(f"✅ Successfully loaded '{target_data_path}'.")
    except FileNotFoundError:
        print(f"❌ Error: Target data file '{target_data_path}' not found."); return
        
    print(f"\n🔄 Aligning target data with the {len(selected_features)} selected features...")
    X_target_selected = X_target[selected_features]
    print(f"✅ Target data aligned. Shape: {X_target_selected.shape}")
    print(f"\n🔄 Predicting with the final {model_name} model...")
    target_predictions_encoded = model.predict(X_target_selected)
    target_predictions_text = le.inverse_transform(target_predictions_encoded)
    predictions_df = pd.DataFrame({'Sample_Index': X_target.index, 'Predicted_Label': target_predictions_text})
    print("\n🎉 --- Target Domain Prediction Complete --- 🎉\nFinal Predictions:")
    print(predictions_df.to_string())
    output_filename = '../results/00_全流程/target_domain_predictions.csv'
    predictions_df.to_csv(output_filename, index=False)
    print(f"\n✅ Predictions have been saved to '{output_filename}'")


# --- 【重新加入】独立的SHAP分析函数 ---
def analyze_with_shap(model, model_name, X_train, data_to_explain, class_names):
    """
    使用SHAP为训练好的模型生成可解释性分析图。
    """
    print("\n" + "="*50)
    print(f"--- Stage 5: Post-hoc Interpretability Analysis with SHAP ---")
    print("="*50)
    print(f"\n🔄 Generating SHAP analysis for {model_name}...")

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(data_to_explain)
        
        # --- 绘制SHAP特征重要性条形图 ---
        print("\n🔄 Generating SHAP Feature Importance Bar Plot...")
        
        # 修改处：添加了max_display参数
        shap.summary_plot(shap_values, 
                          data_to_explain, 
                          plot_type="bar", 
                          class_names=class_names, 
                          show=False, 
                          max_display=50)  # 设置为希望显示的特征数量
        
        fig = plt.gcf()
        ax = plt.gca()
        
        fig.set_size_inches(12, 10)  # 增加图形尺寸以容纳更多特征
        plt.title(f'SHAP Feature Importance for {model_name}')
        
        ax.legend(title='Classes', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        plt.savefig(f'{model_name.replace(" ", "_").lower()}_shap_bar_plot.png', bbox_inches='tight')
        plt.show()
        print(f"✅ SHAP bar plot saved.")
        
    except Exception as e:
        print(f"❌ SHAP analysis failed: {e}")


# --- 主执行函数 ---
if __name__ == "__main__":
    
    # 加载数据
    try:
        with open('../data/processed/processed_data - 2.pkl', 'rb') as f:
            data_bundle = pickle.load(f)
        X_train_full, y_train_text, X_test_full, y_test_text = (
            data_bundle['X_train'], data_bundle['y_train'], 
            data_bundle['X_test'], data_bundle['y_test']
        )
        print(f"✅ Successfully loaded '../data/processed/processed_data.pkl'.")
    except FileNotFoundError:
        print("❌ Error: Data file '../data/processed/processed_data.pkl' not found.")
        exit()

    # 标签编码
    le = LabelEncoder()
    le.fit(pd.concat([y_train_text, y_test_text]).unique())
    y_train_encoded, y_test_encoded = le.transform(y_train_text), le.transform(y_test_text)
    
    # 定义模型
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    }
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # 阶段 1: 特征选择
    model_for_fs = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    X_train_selected, selected_feature_names = select_features_with_rfecv(
        X_train_full, y_train_encoded, model_for_fs, kfold
    )
    X_test_selected = X_test_full[selected_feature_names]

    # 阶段 2: 在筛选后的特征上重新比选模型
    best_model_name_after_fs = compare_models_via_cv(
        X_train_selected, y_train_encoded, models, kfold, title_suffix=" (After Feature Selection)"
    )

    # 阶段 3: 训练、评估并保存最终模型
    final_trained_model = train_and_evaluate_final_model(
        model=models[best_model_name_after_fs],
        model_name=best_model_name_after_fs,
        X_train=X_train_selected,
        y_train=y_train_encoded,
        X_test=X_test_selected,
        y_test=y_test_encoded,
        le=le
    )

    # 阶段 4: 对目标域数据进行预测
    predict_target_domain(
        model=final_trained_model,
        model_name=best_model_name_after_fs,
        target_data_path='../data/processed/processed_data - target.pkl',
        selected_features=selected_feature_names,
        le=le
    )
    
    # **【最终阶段】**: 对最终模型进行SHAP可解释性分析
    analyze_with_shap(
        model=final_trained_model,
        model_name=best_model_name_after_fs,
        X_train=X_train_selected,
        data_to_explain=X_test_selected,
        class_names=le.classes_
    )

#### 蜂窝图

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_selection import RFECV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

# 忽略一些可能出现的计算警告
warnings.filterwarnings('ignore')

# --- 绘图函数 ---
def plot_comparison_chart(results_df, title_suffix=""):
    print(f"\n🔄 Generating model comparison chart{title_suffix}...")
    train_acc = results_df['Train_Accuracy_Mean']
    validation_acc = results_df['Validation_Accuracy_Mean']
    model_names = results_df.index
    x = np.arange(len(model_names)); width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, train_acc, width, label='Train Accuracy')
    rects2 = ax.bar(x + width/2, validation_acc, width, label='Validation Accuracy')
    ax.set_ylabel('Mean Accuracy'); ax.set_title(f'Model Comparison on Train vs. Validation Sets{title_suffix}')
    ax.set_xticks(x); ax.set_xticklabels(model_names); ax.legend()
    ax.bar_label(rects1, padding=3, fmt='%.4f'); ax.bar_label(rects2, padding=3, fmt='%.4f')
    ax.set_ylim(0, 1.1); fig.tight_layout()
    plt.savefig(f'../results/00_全流程/model_comparison_accuracy{title_suffix.replace(" ", "_")}.png')
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    print("\n🔄 Generating confusion matrix chart...")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.5, str(cm[i, j]), ha="center", va="center", color="black", size=10)
    ax.set_title(f'{model_name} Confusion Matrix on Test Set'); ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_confusion_matrix.png')
    plt.show()

def plot_feature_importance(model, feature_names, model_name):
    print(f"\n🔄 Generating feature importance chart for {model_name}...")
    if not hasattr(model, 'feature_importances_'): return
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)
    top_features = feature_importance_df.head(20)
    plt.figure(figsize=(12, 8)); sns.barplot(x='Importance', y='Feature', data=top_features)
    plt.title(f'Top {len(top_features)} Feature Importances for {model_name}'); plt.xlabel('Importance Score'); plt.ylabel('Features')
    plt.tight_layout(); plt.savefig(f'../results/00_全流程/{model_name.replace(" ", "_").lower()}_feature_importance.png')
    plt.show()

# --- 主要实验函数 ---
def select_features_with_rfecv(X_train, y_train, model_for_selection, kfold):
    print("\n--- Stage 1: Wrapper-based Feature Selection ---")
    print(f"\n🔄 Starting RFECV with {type(model_for_selection).__name__}...")
    rfecv = RFECV(estimator=model_for_selection, step=1, cv=kfold, scoring="f1_weighted", min_features_to_select=5, n_jobs=-1)
    rfecv.fit(X_train, y_train)
    print(f"✅ RFECV complete. Optimal number of features found: {rfecv.n_features_}")
    selected_features = X_train.columns[rfecv.support_]
    print(f"   - Selected features are: \n{selected_features.tolist()}")
    X_train_selected = rfecv.transform(X_train)
    return pd.DataFrame(X_train_selected, index=X_train.index, columns=selected_features), selected_features

def compare_models_via_cv(X_train, y_train, models, kfold, title_suffix=""):
    print(f"\n--- Stage 2: Model Comparison{title_suffix} ---")
    results = []
    for model_name, model in models.items():
        cv_results = cross_validate(model, X_train, y_train, cv=kfold, scoring=['accuracy'], n_jobs=-1, return_train_score=True)
        model_result = {'Model': model_name, 'Train_Accuracy_Mean': cv_results['train_accuracy'].mean(), 'Validation_Accuracy_Mean': cv_results['test_accuracy'].mean()}
        results.append(model_result)
    results_df = pd.DataFrame(results).set_index('Model')
    print(f"\nModel Performance Comparison{title_suffix}:\n{results_df.round(4).to_string()}")
    plot_comparison_chart(results_df, title_suffix)
    best_model_name = results_df['Validation_Accuracy_Mean'].idxmax()
    print(f"\n🏆 Based on mean validation accuracy, the best model is: **{best_model_name}**")
    return best_model_name

def train_and_evaluate_final_model(model, model_name, X_train, y_train, X_test, y_test, le):
    print(f"\n--- Stage 3: Final Training and Evaluation ({model_name}) ---")
    model.fit(X_train, y_train)
    print("✅ Final model training complete.")
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n--- Final Model Performance on Test Set ---\n✅ Accuracy: {accuracy:.4f}")
    y_test_text, y_pred_text = le.inverse_transform(y_test), le.inverse_transform(y_pred)
    print("\n📋 Classification Report:"); print(classification_report(y_test_text, y_pred_text))
    plot_confusion_matrix(y_test_text, y_pred_text, class_names=le.classes_, model_name=model_name)
    plot_feature_importance(model, X_train.columns, model_name)
    model_filename = f'../models/00_全流程/{model_name.replace(" ", "_").lower()}_final_model.pkl'
    with open(model_filename, 'wb') as f: pickle.dump(model, f)
    print(f"\n✅ Trained model has been saved to: '{model_filename}'")
    return model

def predict_target_domain(model, model_name, target_data_path, selected_features, le):
    print("\n" + "="*50 + f"\n--- Stage 4: Predicting on Target Domain Data ---")
    try:
        with open(target_data_path, 'rb') as f:
            target_data_bundle = pickle.load(f)
        X_target = target_data_bundle['X_target']
        print(f"✅ Successfully loaded '{target_data_path}'.")
    except FileNotFoundError:
        print(f"❌ Error: Target data file '{target_data_path}' not found."); return
        
    print(f"\n🔄 Aligning target data with the {len(selected_features)} selected features...")
    X_target_selected = X_target[selected_features]
    print(f"✅ Target data aligned. Shape: {X_target_selected.shape}")
    print(f"\n🔄 Predicting with the final {model_name} model...")
    target_predictions_encoded = model.predict(X_target_selected)
    target_predictions_text = le.inverse_transform(target_predictions_encoded)
    predictions_df = pd.DataFrame({'Sample_Index': X_target.index, 'Predicted_Label': target_predictions_text})
    print("\n🎉 --- Target Domain Prediction Complete --- 🎉\nFinal Predictions:")
    print(predictions_df.to_string())
    output_filename = '../results/00_全流程/target_domain_predictions.csv'
    predictions_df.to_csv(output_filename, index=False)
    print(f"\n✅ Predictions have been saved to '{output_filename}'")


# --- 修正后的SHAP分析函数 ---
def analyze_with_shap(model, model_name, X_train, data_to_explain, class_names):
    """
    使用SHAP为训练好的模型生成可解释性分析图。
    生成蜂群图(beeswarm plot)以显示特征影响的方向和大小。
    """
    print("\n" + "="*50)
    print(f"--- Stage 5: Post-hoc Interpretability Analysis with SHAP ---")
    print("="*50)
    print(f"\n🔄 Generating SHAP analysis for {model_name}...")

    try:
        # 创建解释器
        explainer = shap.TreeExplainer(model)
        
        # 计算SHAP值
        shap_values = explainer.shap_values(data_to_explain)
        
        # 获取特征名称
        feature_names = data_to_explain.columns.tolist()
        
        # 检查是多分类还是二分类问题
        if isinstance(shap_values, list):
            # 多分类问题：为每个类别生成一个蜂群图
            print(f"检测到多分类问题，共有 {len(shap_values)} 个类别")
            
            for i, class_name in enumerate(class_names):
                print(f"\n🔄 Generating SHAP beeswarm plot for class '{class_name}'...")
                
                plt.figure(figsize=(12, 10))
                
                # 绘制蜂群图
                shap.summary_plot(
                    shap_values[i], 
                    data_to_explain, 
                    feature_names=feature_names,
                    show=False,
                    max_display=min(20, len(feature_names))  # 限制显示的特征数量
                )
                
                plt.title(f'SHAP Values for {model_name} - Class: {class_name}')
                plt.tight_layout()
                
                # 保存图像
                filename = f'{model_name.replace(" ", "_").lower()}_shap_beeswarm_{class_name}.png'
                plt.savefig(filename, bbox_inches='tight', dpi=300)
                plt.show()
                print(f"✅ SHAP beeswarm plot for class '{class_name}' saved as '{filename}'")
        
        else:
            # 二分类问题
            print(f"\n🔄 Generating SHAP beeswarm plot...")
            
            plt.figure(figsize=(12, 10))
            
            # 绘制蜂群图
            shap.summary_plot(
                shap_values, 
                data_to_explain, 
                feature_names=feature_names,
                show=False,
                max_display=min(20, len(feature_names))  # 限制显示的特征数量
            )
            
            plt.title(f'SHAP Values for {model_name}')
            plt.tight_layout()
            
            # 保存图像
            filename = f'{model_name.replace(" ", "_").lower()}_shap_beeswarm.png'
            plt.savefig(filename, bbox_inches='tight', dpi=300)
            plt.show()
            print(f"✅ SHAP beeswarm plot saved as '{filename}'")
            
    except Exception as e:
        print(f"❌ SHAP analysis failed: {e}")
        import traceback
        traceback.print_exc()


# --- 主执行函数 ---
if __name__ == "__main__":
    
    # 加载数据
    try:
        with open('../data/processed/processed_data - 2.pkl', 'rb') as f:
            data_bundle = pickle.load(f)
        X_train_full, y_train_text, X_test_full, y_test_text = (
            data_bundle['X_train'], data_bundle['y_train'], 
            data_bundle['X_test'], data_bundle['y_test']
        )
        print(f"✅ Successfully loaded '../data/processed/processed_data.pkl'.")
    except FileNotFoundError:
        print("❌ Error: Data file '../data/processed/processed_data.pkl' not found.")
        exit()

    # 标签编码
    le = LabelEncoder()
    le.fit(pd.concat([y_train_text, y_test_text]).unique())
    y_train_encoded, y_test_encoded = le.transform(y_train_text), le.transform(y_test_text)
    
    # 定义模型
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    }
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # 阶段 1: 特征选择
    model_for_fs = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    X_train_selected, selected_feature_names = select_features_with_rfecv(
        X_train_full, y_train_encoded, model_for_fs, kfold
    )
    X_test_selected = X_test_full[selected_feature_names]

    # 阶段 2: 在筛选后的特征上重新比选模型
    best_model_name_after_fs = compare_models_via_cv(
        X_train_selected, y_train_encoded, models, kfold, title_suffix=" (After Feature Selection)"
    )

    # 阶段 3: 训练、评估并保存最终模型
    final_trained_model = train_and_evaluate_final_model(
        model=models[best_model_name_after_fs],
        model_name=best_model_name_after_fs,
        X_train=X_train_selected,
        y_train=y_train_encoded,
        X_test=X_test_selected,
        y_test=y_test_encoded,
        le=le
    )

    # 阶段 4: 对目标域数据进行预测
    predict_target_domain(
        model=final_trained_model,
        model_name=best_model_name_after_fs,
        target_data_path='../data/processed/processed_data - target.pkl',
        selected_features=selected_feature_names,
        le=le
    )
    
    # **【最终阶段】**: 对最终模型进行SHAP可解释性分析
    analyze_with_shap(
        model=final_trained_model,
        model_name=best_model_name_after_fs,
        X_train=X_train_selected,
        data_to_explain=X_test_selected,
        class_names=le.classes_
    )

# 附录，以防需要处理多层级文件夹

# ex

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import scipy.io
import os
import glob

# =============================================================================
# 1. 特征提取核心函数
# =============================================================================
def calculate_time_features(signal: pd.Series) -> dict:
    """计算一维信号的时域特征"""
    signal = signal.dropna()
    rms = np.sqrt(np.mean(signal**2))
    sqrt_amp = (np.mean(np.sqrt(np.abs(signal))))**2
    
    features = {
        'Mean': np.mean(signal), 'RMS': rms, 'Var': np.var(signal),
        'Skew': skew(signal), 'Kurt': kurtosis(signal),
        'CF': np.max(np.abs(signal)) / rms if rms != 0 else 0,
        'MF': np.max(np.abs(signal)) / sqrt_amp if sqrt_amp != 0 else 0,
        'P2P': np.max(signal) - np.min(signal),
    }
    return features

def calculate_freq_features(signal: pd.Series, sampling_rate: int) -> dict:
    """计算一维信号的频域特征"""
    signal = signal.dropna()
    n_points = len(signal)
    if n_points == 0:
        return {'FreqMean': 0, 'FreqSTD': 0, 'FreqSkew': 0, 'FreqKurt': 0}

    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_freq_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_freq_indices]
    amplitudes = np.abs(fft_vals[positive_freq_indices])
    power_spectrum = amplitudes**2
    power_spectrum_normalized = power_spectrum / np.sum(power_spectrum) if np.sum(power_spectrum) != 0 else power_spectrum
    
    freq_mean = np.sum(freqs * power_spectrum_normalized)
    freq_std = np.sqrt(np.sum(((freqs - freq_mean)**2) * power_spectrum_normalized))
    epsilon = 1e-10
    freq_skew = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**3 * power_spectrum_normalized)
    freq_kurt = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**4 * power_spectrum_normalized) - 3

    features = {
        'FreqMean': freq_mean, 'FreqSTD': freq_std,
        'FreqSkew': freq_skew, 'FreqKurt': freq_kurt
    }
    return features

def process_mat_file_to_features(mat_file_path: str, sampling_rate: int = 12000) -> pd.DataFrame:
    """
    加载单个mat文件，提取信号和RPM，并计算所有特征。
    """
    try:
        data = scipy.io.loadmat(mat_file_path)
    except Exception as e:
        print(f"  -> 读取文件 {mat_file_path} 失败: {e}")
        return pd.DataFrame()
        
    # --- 新增代码：提取RPM值 ---
    rpm_value = None
    # 查找以'RPM'结尾的键
    rpm_keys = [key for key in data.keys() if key.endswith('RPM')]
    if rpm_keys:
        # 假设只有一个RPM键，并提取其值
        # .mat文件中的单个数值通常被包裹在多维数组中，例如 [[1797]]
        rpm_value = data[rpm_keys[0]][0][0]
    # --- RPM提取结束 ---

    signal_keys = [key for key in data.keys() if key.endswith('_time')]
    if not signal_keys:
        return pd.DataFrame()
        
    signal_df = pd.DataFrame({key: data[key].flatten() for key in signal_keys})

    results = []
    for col_name in signal_df.columns:
        signal = signal_df[col_name]
        time_feats = calculate_time_features(signal)
        freq_feats = calculate_freq_features(signal, sampling_rate)
        all_feats = {**time_feats, **freq_feats}
        all_feats['Name'] = col_name
        
        # --- 新增代码：将RPM值添加到每一行特征中 ---
        all_feats['RPM'] = rpm_value
        # --- 添加结束 ---

        results.append(all_feats)
        
    return pd.DataFrame(results)

# =============================================================================
# 2. 主程序：遍历、处理、整合
# =============================================================================
def process_all_data(root_dir: str, sampling_rate: int = 12000):
    """
    遍历指定根目录下的所有子文件夹，找到.mat文件，提取特征，并记录文件层级。
    """
    if not os.path.isdir(root_dir):
        print(f"错误：提供的路径 '{root_dir}' 不是一个有效的文件夹。")
        return

    all_rows_list = []
    
    print("开始扫描文件...")
    # Step 1: 遍历所有.mat文件，提取特征并记录原始路径
    for dirpath, _, filenames in os.walk(root_dir):
        # 对文件名进行排序，确保处理顺序一致
        for filename in sorted(filenames):
            if filename.endswith('.mat'):
                full_path = os.path.join(dirpath, filename)
                
                relative_path = os.path.relpath(full_path, root_dir)
                path_parts = relative_path.split(os.sep)
                
                features_df = process_mat_file_to_features(full_path, sampling_rate)
                
                if not features_df.empty:
                    features_df['PathParts'] = [path_parts] * len(features_df)
                    all_rows_list.append(features_df)

    if not all_rows_list:
        print("扫描完成，但在指定文件夹下未找到或未能成功处理任何 .mat 文件。")
        return
        
    # Step 2: 合并所有数据
    final_df = pd.concat(all_rows_list, ignore_index=True)

    # Step 3: 根据路径列表创建层级列
    max_depth = final_df['PathParts'].apply(len).max()
    level_cols = [f'Level_{i+1}' for i in range(max_depth)]
    path_df = pd.DataFrame(final_df['PathParts'].tolist(), index=final_df.index, columns=level_cols)
    final_df = pd.concat([path_df, final_df], axis=1)
    final_df = final_df.drop('PathParts', axis=1)
    
    # Step 4: 处理特殊的"OR"文件夹层级
    print("正在处理'OR'文件夹的特殊层级结构...")
    or_sublevel_col = 'OR_SubLevel'
    final_df[or_sublevel_col] = pd.NA

    fault_type_col = None
    for col in level_cols:
        if 'OR' in final_df[col].astype(str).unique():
            fault_type_col = col
            break
            
    if fault_type_col:
        or_level_index = int(fault_type_col.split('_')[1])
        subsequent_cols = [f'Level_{i+1}' for i in range(or_level_index, max_depth - 1)]

        for i, row in final_df.iterrows():
            if row[fault_type_col] == 'OR' and or_level_index < max_depth:
                or_sub_folder_col = f'Level_{or_level_index+1}'
                if or_sub_folder_col in final_df.columns:
                    final_df.at[i, or_sublevel_col] = row[or_sub_folder_col]
                    
                    for j in range(len(subsequent_cols)):
                        current_col = subsequent_cols[j]
                        next_col_index = or_level_index + j + 2
                        if f'Level_{next_col_index}' in final_df.columns:
                            final_df.at[i, current_col] = row[f'Level_{next_col_index}']
                        else:
                            final_df.at[i, current_col] = pd.NA
    
    # Step 5: 排序
    print("正在对数据进行排序...")
    sort_columns = [col for col in final_df.columns if col.startswith('Level_') or col == or_sublevel_col]
    sort_columns.append('Name')
    final_df = final_df.sort_values(by=sort_columns).reset_index(drop=True)

    # Step 6: 整理最终列顺序并保存
    # --- 更新代码：将RPM添加到特征列列表中 ---
    feature_cols = ['RPM', 'Mean', 'RMS', 'Var', 'Skew', 'Kurt', 'CF', 'MF', 'P2P', 
                    'FreqMean', 'FreqSTD', 'FreqSkew', 'FreqKurt']
    
    final_cols = sort_columns + feature_cols
    final_cols = [col for col in final_cols if col in final_df.columns]
    final_df = final_df[final_cols]

    output_filename = '../data/features/source/all_data_features_sorted_with_rpm.csv'
    final_df.to_csv(output_filename, index=False)
    
    print("\n处理完成！")
    print(f"所有特征已汇总、排序并保存到文件: {output_filename}")
    print("\n最终输出数据预览:")
    print(final_df.head().to_string())


# --- 如何使用 ---
if __name__ == "__main__":
    # ** 请在这里修改为您存放赛方数据的根文件夹路径 **
    root_directory = '../data/raw/source' # '.' 代表当前文件夹

    process_all_data(root_directory)

## Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
import os
from tqdm import tqdm

# =============================================================================
# 1. 核心高级特征计算函数 (逻辑保持不变)
# =============================================================================
def calculate_advanced_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有高级特征（边带+包络），
    并且同时基于DE和FE两种轴承的参数进行计算。
    """
    n_points = len(signal)
    adv_features = {}

    fr = rpm / 60.0
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # Part 1: 边带分析
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])

    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        adv_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        adv_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        adv_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        adv_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # Part 2: 包络解调分析
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return adv_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        adv_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return adv_features

# =============================================================================
# 2. 主执行函数
# =============================================================================
def main():
    """
    主程序：读取CSV，独立遍历文件夹计算新特征，然后拼接。
    """
    # --- 配置区 ---
    input_csv_file = '../data/features/source/dataset.csv'
    # **关键修正1：修正Windows路径的写法，使用正斜杠'/'或双反斜杠'\\'**
    raw_data_root_directory = '../data/raw/source'
    SAMPLING_RATE = 12000
    # --- 配置结束 ---

    # **步骤 1: 读取您已有的数据集**
    if not os.path.isfile(input_csv_file):
        print(f"错误：输入文件 '{input_csv_file}' 不存在。")
        return
    try:
        df_existing = pd.read_csv(input_csv_file)
        print(f"成功读取 '{input_csv_file}'，共 {len(df_existing)} 行数据。")
    except Exception as e:
        print(f"读取CSV文件 '{input_csv_file}' 失败: {e}")
        return

    # **步骤 2: 严格按升序遍历原始数据文件夹，生成信号列表**
    if not os.path.isdir(raw_data_root_directory):
        print(f"错误：原始数据根目录 '{raw_data_root_directory}' 不存在。")
        return
        
    signal_processing_order = []
    print("正在按升序遍历原始数据文件夹...")
    for dirpath, dirnames, filenames in os.walk(raw_data_root_directory):
        dirnames.sort()
        for filename in sorted(filenames):
            if filename.endswith('.mat'):
                full_path = os.path.join(dirpath, filename)
                try:
                    data = scipy.io.loadmat(full_path)
                    signal_keys = sorted([key for key in data.keys() if key.endswith('_time')])
                    for key in signal_keys:
                        signal_processing_order.append({'path': full_path, 'key': key})
                except Exception as e:
                    print(f"读取或解析 {full_path} 时出错: {e}")

    # **步骤 3: 检查顺序和长度是否匹配**
    if len(df_existing) != len(signal_processing_order):
        print("\n警告！！！")
        print(f"您提供的CSV文件行数 ({len(df_existing)}) 与遍历文件夹找到的信号数量 ({len(signal_processing_order)}) 不匹配！")
        print("请检查您的数据和文件夹结构。尽管如此，程序将继续尝试按顺序计算。")

    # **步骤 4: 逐一计算高级特征**
    print("开始计算高级特征...")
    
    # **关键修正2：智能查找RPM列，不再假设列名为'rpm'**
    rpm_col_name = None
    for col in df_existing.columns:
        if col.lower() == 'rpm':
            rpm_col_name = col
            print(f"已自动识别RPM列为: '{rpm_col_name}'")
            break
    
    if rpm_col_name is None:
        print("错误：在 '../data/features/source/dataset.csv' 中未找到RPM列。请确保文件中有一列的名称为'RPM'或'rpm'。")
        return
        
    rpm_values = df_existing[rpm_col_name].tolist()
    new_features_list = []

    for i in tqdm(range(len(signal_processing_order)), desc="计算高级特征"):
        signal_info = signal_processing_order[i]
        rpm = rpm_values[i]
        
        try:
            data = scipy.io.loadmat(signal_info['path'])
            raw_signal = data[signal_info['key']].flatten()
            
            if pd.notna(rpm):
                advanced_features = calculate_advanced_features(raw_signal, rpm, SAMPLING_RATE)
                new_features_list.append(advanced_features)
            else:
                new_features_list.append({})
        except Exception as e:
            print(f"处理信号 {signal_info['key']} (来自 {signal_info['path']}) 时出错: {e}")
            new_features_list.append({})

    # **步骤 5: 合并新特征并保存**
    advanced_features_df = pd.DataFrame(new_features_list)
    final_df = pd.concat([df_existing, advanced_features_df], axis=1)

    output_csv_file = '../data/features/source/dataset_with_added_advanced_features.csv'
    final_df.to_csv(output_csv_file, index=False)

    print(f"\n处理完成！")
    print(f"所有高级特征已添加，并成功保存到新文件: '{output_csv_file}'")
    
    # **关键修正3：使打印预览更健壮，只打印存在的列**
    print("\n最终输出数据预览 (仅显示部分新增列):")
    base_cols_to_preview = [col for col in df_existing.columns[:5] if col in final_df.columns]
    
    # 动态确定RPM和Name列是否存在于最终的DataFrame中
    rpm_and_name_cols = [col for col in [rpm_col_name, 'Name', 'name'] if col in final_df.columns]

    adv_cols_to_preview = []
    if not advanced_features_df.empty:
        adv_cols_to_preview = [col for col in advanced_features_df.columns[:4] if col in final_df.columns]
        
    preview_cols = base_cols_to_preview + rpm_and_name_cols + adv_cols_to_preview
    # 去重
    preview_cols = list(dict.fromkeys(preview_cols))
    
    print(final_df[preview_cols].head().to_string())

if __name__ == "__main__":
    main()